In [12]:

import ee
from utils import *

ee.Authenticate()

# GEE will now use the gcloud identity instead of the separate GEE file
ee.Initialize(project='forestregrowth')

In [ ]:
# move assets between GEE projects

source_folder = 'projects/forestregrowth/assets/CMIP6'
dest_folder = 'projects/climatechange-490617/assets/CMIP6'

def move_gee_folder(source, destination):
    # 1. Create the destination folder if it doesn't exist
    try:
        ee.data.createAsset({'type': 'FOLDER'}, destination)
        print(f"Created folder: {destination}")
    except ee.EEException:
        print(f"Folder {destination} already exists.")

    # 2. List all assets in the source folder
    assets = ee.data.listAssets({'parent': source})['assets']
    
    for asset in assets:
        asset_id = asset['id']
        asset_name = asset_id.split('/')[-1]
        new_path = f"{destination}/{asset_name}"
        
        print(f"Copying {asset_name}...")
        try:
            ee.data.copyAsset(asset_id, new_path)
        except Exception as e:
            print(f"Error copying {asset_name}: {e}")

    print("\nTransfer complete. Verify assets in the Cloud Console before deleting the source.")

# Execute
move_gee_folder(source_folder, dest_folder)

Created folder: projects/climatechange-490617/assets/CMIP6
Copying air_temperature_historical...
Copying air_temperature_ssp126...
Copying air_temperature_ssp245...
Copying air_temperature_ssp585...
Copying moisture_in_upper_portion_of_soil_column_historical...
Copying moisture_in_upper_portion_of_soil_column_ssp126...
Copying moisture_in_upper_portion_of_soil_column_ssp245...
Copying moisture_in_upper_portion_of_soil_column_ssp585...
Copying near_surface_air_temperature_historical...
Copying near_surface_air_temperature_ssp126...
Copying near_surface_air_temperature_ssp245...
Copying near_surface_air_temperature_ssp585...
Copying near_surface_specific_humidity_historical...
Copying near_surface_specific_humidity_ssp126...
Copying near_surface_specific_humidity_ssp245...
Copying near_surface_specific_humidity_ssp585...
Copying precipitation_historical...
Copying precipitation_ssp126...
Copying precipitation_ssp245...
Copying precipitation_ssp585...
Copying surface_downwelling_shortwave

In [14]:

# Deleting assets in bulk from projects. Taken from https://gis.stackexchange.com/questions/467363/batch-deleting-of-earth-engine-assets


def conditional_asset_rm(asset_list, starts_with):
    """Deletes assets from a list if they start with starts_with."""
    success_messages = []
    for asset in asset_list:
        id = asset["id"]
        name = asset["name"]
        findex = 5 if id.startswith("users") else 3
        f = name.split("/")[findex]
        if f.startswith(starts_with):
            ee.data.deleteAsset(id)
            success_messages.append(f"Deleted asset {id}")
    return success_messages


def move_assets_to_folder(starts_with, destination_folder):
    """
    Moves assets from the specified asset list to a given folder 
    if their names start with the specified prefix.

    Args:
        starts_with (str): The prefix string to match.
        destination_folder (str): The folder path to move the assets into.

    Returns:
        list: List of success messages for moved assets.
    """
    success_messages = []
    
    for asset in asset_list:
        id = asset["id"]
        name = asset["name"]
        findex = 5 if id.startswith("users") else 3
        f = name.split("/")[findex]
        
        if f.startswith(starts_with):
            # Define the new asset ID within the destination folder
            new_id = f"{destination_folder}/{f}"
            # Move the asset to the new ID
            ee.data.copyAsset(id, new_id)
            ee.data.deleteAsset(id)
            success_messages.append(f"Moved asset {id} to {new_id}")
    
    return success_messages


In [15]:

asset_list = ee.data.listAssets("projects/forestregrowth/assets/CMIP6")["assets"]
# asset_list


# move_assets_to_folder("near", "projects/amazon-forest-regrowth/assets/mapbiomas")

# conditional_asset_rm(asset_list, "BULCD")

for asset in asset_list:
    id = asset["id"]
    ee.data.deleteAsset(id)